In [ ]:
%cd ..

# maniprot.core

**maniprot.core** is a Python library for Riemannian geometry on protein conformations.
It provides two complementary, GPU-free feature representations for molecular
dynamics (MD) trajectories, both accelerated with [Numba](https://numba.pydata.org)
JIT compilation and multi-core parallelism.

## Overview

maniprot is organised around two complementary approaches to protein conformational
analysis.  Both respect the underlying geometry of molecular structures and
produce representations that can be used with standard machine-learning tools.

| Sub-package | What it models |
|---|---|
| **SO3** | Per-residue orientation (rotation matrices) |
| **pointcloud** | Whole-backbone shape & size (point clouds) |
| **helpers** | Feature extraction from MD trajectories |

In [ ]:
top_pdb = "examples/_structure.pdb"
traj_dcd = "examples/_trajectory.dcd"

In [2]:
import mdtraj as md
from maniprot.core.helpers import lcs, lcs_args
from maniprot.core.SO3 import SO3Manifold
from maniprot.core.pointcloud import PointcloudManifold

In [4]:
# Load trajectory
traj = md.load(traj_dcd, top=top_pdb)

# --- Orientation features (SO3) ---
xyz, n_idx, ca_idx, c_idx = lcs_args(traj)
R = lcs(xyz, n_idx, ca_idx, c_idx)             # (n_frames, n_residues, 3, 3)

so3 = SO3Manifold().fit(R)
R_feat = so3.transform(R)                      # (n_frames, n_residues, 3)

# --- Point-cloud manifold features ---
X = traj.xyz[:, ca_idx, :]                 # (n_frames, n_residues, 3)

pc = PointcloudManifold(delta=0.1).fit(X, learning_rate=1, threshold=1)
X_feat = pc.transform(X)